# Joint mask 更新測試 (雙頭, 3D 版)

輸入**兩個** 3D 頭髮 model（head A = base、head B = donor）和一張 N×N mask（`False` = 長 A 的頭髮、`True` = 長 B 的頭髮），
把兩顆頭**重疊顯示**時會互相交叉的髮絲降到最少。演算法在 [`joint_contour`](../joint_contour/README.md)（[`update_contour`](../update_contour/README.md) 的雙頭版）。

跟 `FM_test_3D.ipynb` 的差別：
- 每根髮絲的狀態是**區域標籤** = 髮根所在格子的 mask 值（True = B 的區域）。A 髮絲標籤為 False、B 髮絲標籤為 True 時才會被畫出來 (visible)。
- **交叉只發生在不同頭的兩根髮絲之間**：同一格柱體裡有一根 visible 的 A 髮絲和一根 visible 的 B 髮絲 = 一個交叉對。
- 三個可選限制（第 3 節的 `NODE_UNIT` / `BALANCE_TOL` / `MOVABLE_ONLY`，預設全部關閉）：
  - `NODE_UNIT="cell"`：FM 節點改成**有髮根的 mask 格子**，一格的標籤同時決定兩顆頭長在這格的所有髮絲，
    每一格永遠只顯示一顆頭。預設 `"strand"` 是每根髮絲各自翻轉，FM 可能把交界處兩顆頭的髮絲都藏起來（交叉變少，但頭皮禿一塊）。
  - `BALANCE_TOL`：True 節點數（B 區域面積）的變動上限；`None` = 不限制。
  - `MOVABLE_ONLY=True`：只翻兩顆頭都有髮根的格子上的節點。
- 只用 FM（active contour 尚未移植）。

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

HERE = "/home/kyh/Desktop/P76154862/highlighting/generate_from_3D_models/scalp_grid_volume_sample"
PROJECT_ROOT = os.path.abspath(os.path.join(HERE, "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import highlighting.generate_from_3D_models.joint_contour as jc
import highlighting.generate_from_3D_models.update_contour as uc
from highlighting.generate_from_3D_models.mask_analysis import plots
from highlighting.generate_from_3D_models.scalp_grid_volume_sample.hair_query import load_strands

plots.setup_cjk_font()

DATASET_ROOT = uc.default_dataset_path()   # <PROJECT_ROOT>/DiffLocks_Dataset/difflocks_dataset
HAIR_A_DIR = os.path.join(DATASET_ROOT, "generated_hairstyles", "base_13_idx_94847")   # head A (base)
HAIR_B_DIR = os.path.join(DATASET_ROOT, "generated_hairstyles", "base_0_idx_77679")    # head B (donor)
STRANDS_NPZ_A = os.path.join(HAIR_A_DIR, "full_strands.npz")
STRANDS_NPZ_B = os.path.join(HAIR_B_DIR, "full_strands.npz")
N = 32  # grid 解析度 (N x N)

## 1. 頭皮 grid 與兩顆頭的髮絲

兩顆頭長在同一個頭皮 mesh 上，共用一個 `ScalpGrid`。把兩顆頭的髮絲疊在同一個 joint index 空間（A 在前、B 在後），
`membership.M` 是 (N², S_a+S_b) 的稀疏矩陣：第 k 根髮絲是否穿過某格上方的柱體。

In [ ]:
grid = jc.ScalpGrid.from_dataset(N)
print(f"有效 cell 數: {grid.grid_valid.sum()} / {grid.grid_valid.size}, half_size = {grid.half_size:.5f}")

pos_a, uv_a, _ = load_strands(STRANDS_NPZ_A)
pos_b, uv_b, _ = load_strands(STRANDS_NPZ_B)
print("head A:", pos_a.shape, " head B:", pos_b.shape)

rowcol_a, rowcol_b = grid.root_rowcol(uv_a), grid.root_rowcol(uv_b)
membership = jc.JointMembership.compute(grid, pos_a, pos_b)
is_b = membership.is_b
count_a, count_b = membership.head_counts()
print(f"incidence nnz = {membership.M.nnz}, 兩顆頭都有髮絲經過的格子 = {int(((count_a > 0) & (count_b > 0)).sum())}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, d, t in zip(axes, [HAIR_A_DIR, HAIR_B_DIR], ["head A", "head B"]):
    ax.imshow(plt.imread(os.path.join(d, "rgb.png"))); ax.set_title(t); ax.axis("off")
axes[2].imshow(np.log1p(uc.uv_to_rc(np.minimum(count_a, count_b).reshape(N, N))), cmap="magma")
axes[2].set_title("log(1 + min(#A, #B)) 每格柱體"); axes[2].axis("off");

## 2. 自訂 N×N mask -> joint 標籤

每根髮絲（兩顆頭都一樣）的標籤 = 髮根所在格子的 mask 值。`visible = (state == is_b)`。

In [ ]:
BASE_COLOR = np.array([0.99, 0.99, 0.99])       # head A 的髮絲
HIGHLIGHT_COLOR = np.array([0.0, 0.0, 0.0])     # head B 的髮絲

mask_matrix = np.zeros((N, N), dtype=bool)
mask_matrix[2:15, 2:15] = True                  # True = head B 的區域

plots.show_color_matrix(jc.mask_to_rgb(mask_matrix, BASE_COLOR, HIGHLIGHT_COLOR))
state0 = jc.grid_mask_to_joint_state(mask_matrix, rowcol_a, rowcol_b)   # (S_a + S_b,) bool, 每根髮絲的標籤
vis0 = jc.visible_strands(state0, is_b)
va, vb = membership.split(vis0)
print(f"visible A: {int(va.sum())} / {va.size}, visible B: {int(vb.sum())} / {vb.size}")

## 3. 更新 mask：joint FM

- `mode="clique"`：最小化 Σ_c na·nb（交叉的 (A, B) 髮絲對數）；`"cutnet"`：最小化 na>0 且 nb>0 的格數。
- `lam`：離開初始標籤的懲罰（節點裡每根髮絲算一次）。
- `iter_nodes[0]` 是初始節點標籤，之後每個元素是一個 pass 的結果；`iter_states` 是展開到每根髮絲的標籤
  （`NODE_UNIT="strand"` 時兩者相同）。

In [ ]:
# lam 的尺度跟柱體大小有關 (N=32 的柱體比 N=256 大 64 倍, 交叉對數也大得多)。目標函數沒有「區域要連在一起」的項，
# lam 太小時 FM 會把 B 區域整塊搬到 B 髮絲不跟 A 交叉的地方 (髮旋分線、頭皮邊緣)；N=32 時 lam≈30000 才會留在原本的方塊附近。
lam = 30000

# 可選限制 (預設全部關閉)
NODE_UNIT = "strand"    # "strand": 每根髮絲一個節點；"cell": 每個有髮根的格子一個節點
BALANCE_TOL = None      # None = 不限制；例如 0.02 = True 節點數最多變動 2%
MOVABLE_ONLY = False    # True = 只翻兩顆頭都有髮根的格子上的節點

incidence = jc.build_joint_incidence(membership, rowcol_a, rowcol_b, node_unit=NODE_UNIT)
print(f"FM 節點 ({NODE_UNIT}) = {incidence.n_nodes}, 兩顆頭都有髮絲經過的柱體 = {incidence.cell_ptr.shape[0] - 1}")

node0 = incidence.node_state_from_mask(mask_matrix)
iter_nodes = jc.joint_fm_refine(incidence, node0, mode="clique", lam=lam, max_passes=3, balance_tol=BALANCE_TOL,
                                movable_only=MOVABLE_ONLY, return_history=True)
iter_states = [incidence.strand_state(x) for x in iter_nodes]

n_iter = len(iter_nodes) - 1
print("每輪被翻轉的節點比例:", jc.change_ratios(iter_nodes))
for i, (x, s) in enumerate(zip(iter_nodes, iter_states)):
    cells, pairs = jc.joint_fm_objectives(incidence, x)
    va, vb = membership.split(jc.visible_strands(s, is_b))
    print(f"iteration {i}: conflict cells = {cells}, crossing pairs = {pairs}, "
          f"visible A = {int(va.sum())}, visible B = {int(vb.sum())}")

## 4. 結果分析

In [ ]:
# 髮根 UV 點雲：只畫 visible 的髮絲，A = 淺色、B = 黑色 (重疊顯示的結果)
def plot_joint_uv(states, max_pts=40000, seed=0):
    rng = np.random.default_rng(seed)
    uv = np.concatenate([uv_a, uv_b])
    fig, axes = plt.subplots(len(states), 1, figsize=(16, 4 * len(states)), squeeze=False)
    for i, (ax, s) in enumerate(zip(axes[:, 0], states)):
        vis = np.nonzero(jc.visible_strands(s, is_b))[0]
        vis = rng.choice(vis, min(max_pts, vis.size), replace=False)
        ax.set_facecolor("gray")
        ax.scatter(uv[vis, 0], uv[vis, 1], c=jc.joint_colors(is_b[vis], BASE_COLOR, HIGHLIGHT_COLOR), s=3, linewidths=0)
        ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_aspect("equal")
        ax.set_title(f"visible 髮根 (iteration {i}): A 淺色 / B 黑色")
    return fig

plot_joint_uv(iter_states);

In [ ]:
# 每格柱體的交叉髮絲對數 na * nb (log 顯示)
pair_maps = [np.log1p(jc.conflict_pairs_map(s, membership)) for s in iter_states]
plots.plot_grid_score_iterations(pair_maps, label="log(1 + 交叉對數)", vmax=max(float(m.max()) for m in pair_maps));

In [ ]:
# 更新後的 N×N mask：有髮根的格子取節點標籤的多數決，其他格子看鄰近格子 (JointIncidence.mask_from_node_state)
grid_masks = [incidence.mask_from_node_state(x, fallback=mask_matrix, movable_only=MOVABLE_ONLY) for x in iter_nodes]
print("mask 被改動的格數:", [int((m != mask_matrix).sum()) for m in grid_masks])
plots.plot_grid_mask_iterations(grid_masks, BASE_COLOR, HIGHLIGHT_COLOR);

In [ ]:
# 一行版 (同一組參數)，結果應與上面最後一輪相同
out = jc.update_joint_mask(pos_a, uv_a, pos_b, uv_b, mask_matrix, grid=grid,
                           membership_a=membership.M[:, :membership.n_a], membership_b=membership.M[:, membership.n_a:],
                           method_kwargs=dict(mode="clique", lam=lam, max_passes=3, node_unit=NODE_UNIT,
                                              balance_tol=BALANCE_TOL, movable_only=MOVABLE_ONLY), verbose=False)
print(out["objectives"], " 與逐步版相同:", np.array_equal(out["binary_mask"], grid_masks[-1]))

## 5. Trim 與 Blender 渲染

`joint_trim`：挑出 visible 且跟另一顆頭交叉最多的 `max_cut` 根髮絲，渲染 `*_trimmed` 版本時整根不畫。
渲染時兩顆頭合併成一個 npz（`output/joint/joint_strands.npz`），hidden 的髮絲放進 ignore mask；A 畫成 BASE_COLOR、B 畫成 HIGHLIGHT_COLOR。

In [ ]:
trimmed_masks = [jc.joint_trim(s, membership, max_cut=1000) for s in iter_states]

In [ ]:
RUN_RENDER = False

if RUN_RENDER:
    from highlighting.generate_from_3D_models.joint_contour import render as jrender
    out_root = os.path.join(HERE, "output", "joint")
    joint_npz, _ = jrender.write_joint_strands_npz(STRANDS_NPZ_A, STRANDS_NPZ_B, os.path.join(out_root, "joint_strands.npz"))
    jrender.render_joint_iterations(
        iter_states, is_b, out_root, joint_npz=joint_npz, dataset_path=DATASET_ROOT, grid_size=N,
        trim_masks=trimmed_masks, base_color=BASE_COLOR, highlight_color=HIGHLIGHT_COLOR,
        source="joint_contour_updating_test.ipynb",
    )